In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, f1_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, TensorDataset, DataLoader
from tab_transformer_pytorch import TabTransformer, FTTransformer
from preprocessing import get_features_and_target_classification, get_features_and_target
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from RMSELoss import RMSELoss
import plotly.graph_objects as go
from tabpfn import TabPFNClassifier, TabPFNRegressor
from tabpfn.constants import ModelVersion
from sklearn.model_selection import train_test_split

In [2]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [ ]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data_classifier.csv")
dev_df = pd.read_csv("data/development_data_classifier.csv")

sc = MinMaxScaler()

target_column = ("Category")  

x_train, y_train = get_features_and_target_classification(train_df, target_column)
x_dev, y_dev = get_features_and_target_classification(dev_df, target_column)


x_train = sc.fit_transform(X=x_train)
x_dev = sc.transform(x_dev)

In [4]:
x_train

,Sample ID,Pressure (PSI),Welding Time (ms),Angle (Deg),Force (N),Current (A),Thickness A (mm),Thickness B (mm)
0,1,35,200,0,0.00,1315.41,0.922,0.920
1,1,35,200,0,3.41,1337.45,0.922,0.920
2,1,35,200,0,6.82,1081.47,0.922,0.920
3,2,35,1500,0,0.00,1819.13,0.920,0.925
4,2,35,1500,0,3.41,2016.44,0.920,0.925
...,...,...,...,...,...,...,...,...
2448,494,60,1200,0,96.62,3400.24,0.620,0.624
2449,494,60,1200,0,96.55,3855.83,0.620,0.624
2450,494,60,1200,0,96.48,4173.30,0.620,0.624
2451,494,60,1200,0,96.41,3485.17,0.620,0.624


# Add Physical Columns Interfacial_Failure and Pullout_Failure

In [5]:
def compute_interfacial_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     f_pull = 1 * (np.pi/4) * (4 * np.sqrt(t))**2 * (0.7 * 365) 
     return np.round(f_pull, 1) 

def compute_pullout_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     x = df[['Thickness A (mm)', 'Thickness B (mm)']].sum(axis=1)
     # x can be approximated to metal sheet thickness. Change 2*t either to t to use the thinner 
     # metal sheet or 2*x to test if the sum of both metal sheets give beter results
     f_pull = np.pi * ((4 * np.sqrt(t)) + 2*t)*t*365 
     return np.round(f_pull, 1) 

In [6]:
x_dev

,Sample ID,Pressure (PSI),Welding Time (ms),Angle (Deg),Force (N),Current (A),Thickness A (mm),Thickness B (mm)
0,6,95,1500,0,0.00,1213.38,0.918,0.925
1,6,95,1500,0,8.29,1230.27,0.918,0.925
2,6,95,1500,0,16.57,1217.63,0.918,0.925
3,6,95,1500,0,24.85,1111.28,0.918,0.925
4,6,95,1500,0,33.15,1126.65,0.918,0.925
...,...,...,...,...,...,...,...,...
835,489,60,1200,0,98.26,3576.75,0.625,0.624
836,489,60,1200,0,98.22,3024.94,0.625,0.624
837,489,60,1200,0,98.18,2984.87,0.625,0.624
838,489,60,1200,0,98.14,2950.68,0.625,0.624


# Fit Model

In [7]:
from sklearn.metrics import accuracy_score, roc_auc_score

# columns that vary within a sample
time_series_cols = ["Force (N)", "Current (A)"]

# static columns
static_cols = ["Pressure (PSI)", "Welding Time (ms)", "Angle (Deg)", "Thickness A (mm)", "Thickness B (mm)"]

agg_df_train = train_df.groupby("Sample ID")[time_series_cols].agg(
    ['mean', 'std', 'min', 'max']
)
agg_df_dev = dev_df.groupby("Sample ID")[time_series_cols].agg(
    ['mean', 'std', 'min', 'max']
)

# flatten multi-index columns
agg_df_train.columns = ['_'.join(col) for col in agg_df_train.columns]
agg_df_dev.columns = ['_'.join(col) for col in agg_df_dev.columns]

# --- Add static columns (take first value per sample) --- 
static_df_train = train_df.groupby("Sample ID")[static_cols].first() 
static_df_dev = dev_df.groupby("Sample ID")[static_cols].first() 

 # Total Welding Time, adds the Welding Time Cycles in a single sample
weld_time_sum_train = train_df.groupby("Sample ID")["Welding Time (ms)"].sum() 
weld_time_sum_train = weld_time_sum_train.rename("Welding_Time_Total") 
weld_time_sum_dev = dev_df.groupby("Sample ID")["Welding Time (ms)"].sum() 
weld_time_sum_dev = weld_time_sum_dev.rename("Welding_Time_Total") 

# Combination of datasets
join_static_df_train = agg_df_train.join(static_df_train) 
x_train = join_static_df_train.join(weld_time_sum_train) 
join_static_df_dev = agg_df_dev.join(static_df_dev) 
x_dev = join_static_df_dev.join(weld_time_sum_dev) 

# Target value
y_train = train_df.groupby("Sample ID")['Category'].first()

classifier = TabPFNClassifier() 

classifier.fit(x_train,y_train)

predictions_class_train = classifier.predict(x_train)
predictions_class_dev = classifier.predict(x_dev)

In [8]:
#y = y.to_frame(name="Category")   # ensure y is a DataFrame
#x["Pullforce"] = None


# Add Pullforce as Target

In [9]:
def compute_pullforces(x, y, predictions_class):
    pullforces = []

    for sample_id, pred in zip(y.index, predictions_class):
        row_x_df = x.loc[[sample_id]]  # 1-row DataFrame

        if pred == "Bad":
            value = compute_interfacial_failure(row_x_df).iloc[0]
        else:
            value = compute_pullout_failure(row_x_df).iloc[0]

        pullforces.append(value)

    return np.array(pullforces)


In [10]:
y_train = train_df.groupby("Sample ID")['Category'].first()
predictions_class_train = classifier.predict(x_train)

pullforces_train = compute_pullforces(x_train, y_train, predictions_class_train)

y_dev = dev_df.groupby("Sample ID")['Category'].first()
predictions_class_dev = classifier.predict(x_dev)

pullforces_dev = compute_pullforces(x_dev, y_dev, predictions_class_dev)


In [11]:
print(predictions_class_dev)

['Good' 'Bad' 'Good' 'Bad' 'Good' 'Bad' 'Good' 'Bad' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good']


In [12]:
print(pullforces_dev)

[5967.  2960.3 6042.8 2976.3 6097.2 2976.3 6097.2 3018.1 3423.9 3170.5
 3245.7 3237.3 3254.1 3304.7 3195.5 3245.7 3153.9 3203.8 3087.8 3330.1
 3245.7 3270.9 3162.2 3187.1 3220.5 3220.5 3145.6 3187.1 3129.  3162.2
 3096.  3112.5 3220.5 3228.9 3096.  3254.1 3270.9 3137.3 3170.5 3212.2
 3137.3 3254.1 3228.9 3145.6 3212.2 3220.5 3254.1 3262.5 3145.6 3071.3
 3055.  3038.6 3178.8 3129.  3162.2 3145.6 3170.5 3120.8 3162.2 3178.8
 3153.9 3104.3 3137.3 3153.9 3129.  3137.3 3096.  3055.  3079.6 3137.3
 3104.3 3220.5 3162.2 3178.8 3112.5 3104.3 3137.3 3104.3 3129.  3187.1
 3212.2 3145.6 3195.5 3170.5 3170.5 3112.5 3120.8 3162.2 3153.9 3170.5
 3038.6 3112.5 3153.9 3071.3 3153.9 3145.6 3170.5 3112.5 3153.9]


In [13]:
y_dev_Regressor = dev_df.groupby("Sample ID")['PullTest (N)'].first()
y_dev_delta = y_dev_Regressor - pullforces_dev 

y_train_Regressor = train_df.groupby("Sample ID")['PullTest (N)'].first()
y_train_delta = y_train_Regressor - pullforces_train

In [14]:
y_dev_Regressor.head(10)

Sample ID
6     4161.4
8     1836.4
14    3970.1
16    2509.8
22    4952.0
25    2867.4
28    3855.5
32    2764.4
34    3281.1
40    3256.1
Name: PullTest (N), dtype: float64

In [15]:
y_dev_delta.head(10)

Sample ID
6    -1805.6
8    -1123.9
14   -2072.7
16    -466.5
22   -1145.2
25    -108.9
28   -2241.7
32    -253.7
34    -142.8
40      85.6
Name: PullTest (N), dtype: float64

In [16]:
x_train

,Force (N)_mean,Force (N)_std,Force (N)_min,Force (N)_max,Current (A)_mean,Current (A)_std,Current (A)_min,Current (A)_max,Pressure (PSI),Welding Time (ms),Angle (Deg),Thickness A (mm),Thickness B (mm),Welding_Time_Total
Sample ID,,,,,,,,,,,,,,
1,3.410000,3.410000,0.00,6.82,1244.776667,141.856410,1081.47,1337.45,35,200,0,0.922,0.920,600
2,25.967500,16.599982,0.00,52.25,1840.571250,98.110791,1709.96,2016.44,35,1500,0,0.920,0.925,24000
4,8.286667,8.285001,0.00,16.57,1416.673333,90.008858,1321.93,1501.05,95,200,0,0.912,0.924,600
5,33.136667,8.285001,24.85,41.42,1474.350000,182.155941,1268.82,1615.83,95,200,0,0.948,0.939,600
7,97.644667,21.377203,63.82,127.46,1035.198000,92.864686,871.91,1150.82,35,1500,0,0.930,0.937,22500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
488,98.536154,0.083520,98.34,98.63,3916.430000,529.220757,2730.90,4554.81,60,1200,0,0.625,0.622,15600
490,97.889231,0.283092,97.32,98.15,2803.468462,276.735971,2542.37,3369.76,60,1200,0,0.622,0.632,15600
492,98.490833,0.277700,98.07,98.87,3625.364167,488.348727,2495.94,4337.00,60,1200,0,0.666,0.633,14400


# Fit 2nd Model

In [17]:
# Initialize the regressor
regressor = TabPFNRegressor()  # Uses TabPFN-2.5 weights, trained on synthetic data only.
# To use TabPFN v2:
# regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)

regressor.fit(x_train, y_train_delta)

# Predict on the test set
predictions_class_dev = regressor.predict(x_dev)
final_pred = pullforces_dev + predictions_class_dev

# Check Validation Data

In [18]:
y_dev = dev_df.groupby("Sample ID")['PullTest (N)'].first()
y_dev

# Convert to numpy arrays (if not already)
true_vals = np.array(y_dev).ravel()
pred_vals = np.array(final_pred).ravel()

# Sample index for plotting
sample_idx = np.arange(len(true_vals))

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=sample_idx, y=true_vals, mode="markers",
    name="Original Values", marker=dict(color="red", size=6)
))

fig.add_trace(go.Scatter(
    x=sample_idx, y=pred_vals, mode="markers",
    name="Predicted Values", marker=dict(color="blue", size=6)
))

# Connecting lines (one per sample) 
for i in range(len(sample_idx)): 
    fig.add_trace(go.Scatter( 
        x=[sample_idx[i], sample_idx[i]], 
        y=[true_vals[i], pred_vals[i]], 
        mode="lines", 
        line=dict(color="gray", width=1), 
        showlegend=False
    ))

fig.update_layout(
    title="Validation Samples: True vs Prediction (TabPFN)",
    xaxis_title="Sample Index",
    yaxis_title="Pull Force",
    template="seaborn"
)

fig.write_html("Graphs/TabPFN_Evaltrue.html")
fig.show()


In [19]:
y_dev.tail(94)

Sample ID
25     2867.4
28     3855.5
32     2764.4
34     3281.1
40     3256.1
        ...  
468    2865.5
471    2860.7
474    2902.6
475    3000.7
489    2054.5
Name: PullTest (N), Length: 94, dtype: float64

In [20]:
final_pred

array([4129.01049805, 2154.22694092, 4184.3168457 , 2165.96094971,
       5337.92790527, 2256.78168945, 5332.00633545, 2263.8130127 ,
       3525.85825195, 3249.05647278, 3342.02723999, 2912.22895508,
       2927.26470337, 2956.58165283, 2839.68847656, 2875.87687988,
       2790.0239624 , 2821.02351074, 2665.39881592, 2824.84829102,
       2737.91429443, 2770.62137451, 2698.19365234, 2719.21383057,
       2730.40707397, 2751.21612549, 2695.94002686, 2740.12844238,
       2753.15304565, 2742.97703857, 2718.25714111, 2728.02301025,
       2774.65332031, 2764.3911499 , 2672.21069336, 2809.94777832,
       2826.95352783, 2738.5532959 , 2740.49499512, 2791.2112915 ,
       2718.49320679, 2828.18877563, 2779.15720215, 3089.00480804,
       3135.99098511, 3128.49562836, 3157.78511963, 3168.04357147,
       3065.74946747, 2997.2654007 , 2993.36989975, 2975.33297501,
       3095.30749207, 2973.15783691, 3071.84168549, 3049.80729065,
       3056.62919617, 3013.63912659, 3034.9541275 , 3054.60534

# Check Validation Loss and R2

In [21]:
# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev, final_pred)
rmse = np.sqrt(mean_squared_error(y_dev, final_pred))
R2   = r2_score(y_dev, final_pred)


print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")

MAE:  133.51
RMSE: 231.85
R2: 0.58
